# 第26章　医療AIのための統計の基礎 ― 数字を正しく語る**『医療診断支援AIの社会実装（社会実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-social

## 検定を、正しく選ぶ

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar# b: v1正解かつv2誤り, c: v1誤りかつv2正解 の症例数result = mcnemar([[a, b], [c, d]], exact=True)   # bとcの偏りを検定

## 分布を仮定しない ― ノンパラメトリック検定とブートストラップ

In [ ]:
import numpy as npdef needs_both_classes(y_idx):                 # ROC-AUCなど、両クラスが必要な指標用    return len(np.unique(y_idx)) >= 2def bootstrap_ci(y, p, metric, groups=None, n_boot=2000, seed=0, valid=needs_both_classes):    """groups: 行ごとの患者ID（施設単位で抜くなら施設ID）。省略時は 1行＝1独立患者 とみなす。    複数画像・複数読影・施設内相関があるデータに、行単位の抽出をそのまま使ってはいけない。    valid: その再標本で指標が定義できるかを判定する関数。既定はAUC向け（両クラス必要）。    正解率のように単一クラスでも定義できる指標には valid=lambda y_idx: True を渡す。    感度（陽性=1）のように陽性が要る指標には valid=lambda y_idx: np.any(y_idx == 1) を渡す    （陽性ゼロの再標本をライブラリ既定の 0 点として集計しない）。"""    rng = np.random.default_rng(seed)    y, p = np.asarray(y), np.asarray(p)    groups = np.arange(len(y)) if groups is None else np.asarray(groups)    uniq = np.unique(groups)    rows_of = {g: np.where(groups == g)[0] for g in uniq}    scores = []    for _ in range(n_boot):        picked = rng.choice(uniq, size=len(uniq), replace=True)       # 患者（群）単位で復元抽出        idx = np.concatenate([rows_of[g] for g in picked])            # その患者の行をまとめて複製        if not valid(y[idx]):                                         # 指標が定義できない回は除いて数える（規約を先に決めておく）            continue        v = metric(y[idx], p[idx])        if np.isfinite(v):                                            # NaN/inf を返した回も除く            scores.append(v)    if len(scores) == 0:                                              # 全回が無効なら、区間を作らず「評価不能」を返す        return None, 0    ci = np.percentile(scores, [2.5, 97.5])                           # 分布仮定なしの95%CI    return ci, len(scores)                                            # 有効反復数も一緒に返して報告する

## 相関と因果を、取り違えない ― 交絡とショートカット

In [ ]:
# 交絡(重症度)で層別してから効果を見る、の骨格for stratum, sub in df.groupby("severity"):    rate_t = sub[sub.treated == 1]["bad_outcome"].mean()    rate_c = sub[sub.treated == 0]["bad_outcome"].mean()    print(stratum, rate_t - rate_c)   # 測定した重症度で層別した比較。未測定・残余の交絡は除けない